# Basics and Async Concepts 

### Generators

They are just like the normal functions but with the capability to pause using "yield" keyword.

In [1]:
from time import sleep

In [2]:
# Generator example

def timer(number):
    while number > 0:
        yield number
        number -= 1
        sleep(1)

In [3]:
timer_gen_obj = timer(10)

In [4]:
timer_gen_obj.send(None)

10

In [5]:
timer_gen_obj.__next__()

9

In [6]:
for i in timer_gen_obj:
    print(i)

8
7
6
5
4
3
2
1


In [7]:
# Corroutine like generator

def is_this(word):
    while True:
        user_sentence = yield 
        if user_sentence == "exit":
            break
        if word in user_sentence:
            print(True)
        else:
            print(False)
        

In [8]:
# To start a generator (coroutine type), either use .send(None) or next() ONCE.
find_yo_gen_obj = is_this("yo")
type(find_yo_gen_obj)  # generator
find_yo_gen_obj.send(None)  # or we can simply write next(k_gen_obj) to run and get to the yield statement


In [9]:
find_yo_gen_obj.send("ksdhkfh sjdhfkhsdkh k dskjhfkhkh sdk yo fhkj ksdhj")
        

True


## Async keyword

In [10]:
#  regular func vs async func

def  hello_name(name):
    print(f"hello {name}")

hello_name

<function __main__.hello_name(name)>

In [11]:
async def  async_hello_name(name):
    print(f"hello {name}")

async_hello_name

<function __main__.async_hello_name(name)>

In [12]:
hello_name("test")

hello test


In [13]:
async_hello_name("test")

<coroutine object async_hello_name at 0x000001CFD5ACA810>

#### all async functions require something else to Run it. They can't run themselves.

In [14]:
async_cor_obj = async_hello_name("test")

In [15]:
# async_cor_obj.send(None)

In [16]:
def run(coroutine):
    try:
        coroutine.send(None)
    except Exception as e:
        return e.value    

In [17]:
run(async_cor_obj)
run(async_hello_name("test"))


hello test
hello test


In [18]:
run(async_hello_name("testttttt"))

hello testttttt


In [19]:
# running the async function (coroutine) using the await keyword (which is recommended instead of coroutine.send())

await async_hello_name("tom")

hello tom


### some examples


In [36]:
import asyncio
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor



In [23]:

def fetch_data(param):
    print(f"Do something with {param}...", flush=True)
    time.sleep(param)
    print(f"Done with {param}", flush=True)
    return f"Result of {param}"



In [12]:
fetch_data(1)

Do something with 1...


'Result of 1'

In [ ]:
asyncio.create_task(asyncio.to_thread(fetch_data, 1)) 

<Task pending name='Task-9' coro=<to_thread() running at d:\pyinstallfolder\py312\Lib\asyncio\threads.py:12>>

Do something with 1...


In [ ]:
asyncio.create_task(asyncio.to_thread(fetch_data, 2))

<Task pending name='Task-10' coro=<to_thread() running at d:\pyinstallfolder\py312\Lib\asyncio\threads.py:12>>

Do something with 2...


In [ ]:
task1 = asyncio.create_task(asyncio.to_thread(fetch_data, 1)) 

Do something with 1...


Done with 1


In [ ]:
task2 = asyncio.create_task(asyncio.to_thread(fetch_data, 2))

Do something with 2...


In [26]:
await task1

'Result of 1'

In [31]:
loop = asyncio.get_running_loop()
loop

<_WindowsSelectorEventLoop running=True closed=False debug=False>

In [32]:
# Create a process pool executor context
with ProcessPoolExecutor() as executor:
    # Schedule fetch_data(1) to run in a separate process
    task1 = loop.run_in_executor(executor, fetch_data, 1)
    task1
    result1 = await task1
    print("Process 1 fully completed")

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [37]:

async def main() -> list:
    """
    Asynchronous main function demonstrating running blocking code in threads and processes.
    Returns:
        list: A list containing the results from the last two fetch_data calls (from the process pool).
    """
    # --- Run in Threads ---
    # Schedule fetch_data(1) to run in a separate thread and wrap it as an asyncio Task, as well as starts executing it,
    task1 = asyncio.create_task(asyncio.to_thread(fetch_data, 1)) 
    # Schedule fetch_data(2) to run in a separate thread and wrap it as an asyncio Task
    task2 = asyncio.create_task(asyncio.to_thread(fetch_data, 2))
    # Await the result of the first thread task
    result1 = await task1
    print("Thread 1 fully completed")
    # Await the result of the second thread task
    result2 = await task2
    print("Thread 2 fully completed")



    ########################################################################################################################
    # --- Run in Process Pool ---
    # Get the current running event loop
    loop = asyncio.get_running_loop()

    # Create a process pool executor context
    with ThreadPoolExecutor() as executor:
        # Schedule fetch_data(1) to run in a separate process
        task1 = loop.run_in_executor(executor, fetch_data, 1)
        # Schedule fetch_data(2) to run in a separate process
        task2 = loop.run_in_executor(executor, fetch_data, 2)

        # Await the result of the first process task
        result1 = await task1
        print("Process 1 fully completed")
        # Await the result of the second process task
        result2 = await task2
        print("Process 2 fully completed")

    # Return the results from the process pool executions as a list
    return [result1, result2]

In [38]:
main()

<coroutine object main at 0x0000024D8AC3B940>

In [39]:
# results = asyncio.run(main()) ### This won't work in jupyter notebook
results = await main()
print(results)

Do something with 1...
Do something with 2...


In [40]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor  # Changed from ProcessPoolExecutor

def fetch_data(param):
    print(f"Do something with {param}...", flush=True)
    time.sleep(param)
    print(f"Done with {param}", flush=True)
    return f"Result of {param}"

async def main():
    # Run in Threads
    task1 = asyncio.create_task(asyncio.to_thread(fetch_data, 1))
    task2 = asyncio.create_task(asyncio.to_thread(fetch_data, 2))
    result1 = await task1
    print("Thread 1 fully completed")
    result2 = await task2
    print("Thread 2 fully completed")

    # Run in Thread Pool (instead of Process Pool)
    loop = asyncio.get_running_loop()

    with ThreadPoolExecutor() as executor:  # Changed from ProcessPoolExecutor
        task1 = loop.run_in_executor(executor, fetch_data, 1)
        task2 = loop.run_in_executor(executor, fetch_data, 2)

        result1 = await task1
        print("Thread Pool 1 fully completed")
        result2 = await task2
        print("Thread Pool 2 fully completed")

    return [result1, result2]

# Run the async function
results = await main()
print(results)

Do something with 1...Do something with 2...

